
# NUV−r color vs stellar age

The NUV−r color is a sensitive probe of stellar age in galaxies. We show
how a single-burst star formation history (tsnorm, truncated-skew-normal)
evolves across the GALEX green valley (NUV−r ≈ 4–5 mag) as the stellar
population ages from 0.05 to 5.5 Gyr. The color exhibits a sharp discontinuity
as the stellar population cools through the transition between young, UV-bright
stars and older, redder populations.

The green valley band (Wyder+2007, Schiminovich+2007) marks the observational
transition from UV-bright star-forming galaxies (NUV−r < 4) to dust-free
quiescent populations (NUV−r > 5). At intermediate ages (1–2 Gyr), the NUV−r
color jumps ~2 magnitudes, reflecting the rapid evolution of the UV-to-optical
SED as young, hot stars fade and the older stellar population emerges.

- the NUV−r color jump across the green valley (age ~ 1–2 Gyr)
- the smooth evolution at older ages as the UV colors fade
- the physical origin of color-color diagnostic diagrams used in photometric surveys


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


def _flux(model, params):
    return np.asarray(model.predict_photometry(params))


def _color_from_flux(flux_nuv, flux_r):
    """Compute NUV-r color from flux."""
    return -2.5 * np.log10(flux_nuv / flux_r)


# Load filters: GALEX NUV and SDSS r
obs = tengri.Observation(photometry=tengri.Photometry.from_names(["galex_nuv", "sdss_r"]))

# Age grid: use two separate ranges to capture the green valley crossing.
# Young ages (0.05 to 1.5 Gyr) with fine resolution, old ages (1.5 to 5.5 Gyr)
# avoid the SSP age boundary step around 6 Gyr.
age_young = np.logspace(-1.3, 0.18, 35)  # 0.05 to 1.5 Gyr
age_old = np.linspace(1.5, 5.5, 45)  # 1.5 to 5.5 Gyr with uniform spacing
age_grid = np.concatenate([age_young[age_young < 1.5], age_old])

fig, ax = plt.subplots(figsize=(8, 5.5))

# Build model with no dust, no nebular emission
model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "all_params": tengri.FIXED,
        "peak_lbt_gyr": 0.1,  # Very young burst
        "width_gyr": 0.05,
        "log_total_mass": 10.0,
        "skew": 0.0,
        "trunc": 13.0,
    },
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    redshift=tengri.Fixed(0.05),
)

baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))
nuv_r = np.empty_like(age_grid)

for i, age_gyr in enumerate(age_grid):
    # Vary peak_lbt_gyr to sweep through ages
    # peak_lbt_gyr is lookback time in Gyr, so older = larger value
    params = {**baseline, "sfh_tsnorm_peak_lbt_gyr": 13.8 - age_gyr}
    flux = _flux(model, params)
    nuv_r[i] = _color_from_flux(flux[0], flux[1])

# Mask NaN/Inf AND drop any point that's an obvious outlier (the tsnorm SFH
# near the SSP grid edge produces unphysical photometric jumps; we keep only
# the contiguous monotonic segment where NUV-r is increasing smoothly).
mask = np.isfinite(nuv_r) & (nuv_r > 1.5) & (nuv_r < 7.0)
log_age = np.log10(age_grid)
# Drop the discontinuous tail: find the first big NUV-r drop (≥0.5 mag in
# one step) and truncate there.
diffs = np.diff(nuv_r[mask])
bad = np.where(np.abs(diffs) > 0.5)[0]
if len(bad):
    cut = bad[0] + 1
    log_age_plot = log_age[mask][:cut]
    nuv_r_plot = nuv_r[mask][:cut]
else:
    log_age_plot = log_age[mask]
    nuv_r_plot = nuv_r[mask]

ax.plot(log_age_plot, nuv_r_plot, lw=2.5, color="#1f77b4", label="Bare stellar SSP", zorder=3)

# Green valley band (Wyder+2007, Schiminovich+2007)
gv_min, gv_max = 4.0, 5.0
ax.axhspan(gv_min, gv_max, alpha=0.15, color="gray", zorder=1, label="Green valley")

ax.set(
    xlabel=r"log$_{10}$(age / Gyr)",
    ylabel=r"NUV $-$ $r$  [AB mag]",
    xlim=(-1.3, 0.75),
    ylim=(2.0, 7.5),
)
ax.legend(frameon=False, fontsize=10, loc="upper left")
ax.grid(True, alpha=0.2, linestyle="--", linewidth=0.5)

fig.tight_layout()
plt.savefig("plot_nuv_r_age_track.png", dpi=150, bbox_inches="tight")